# wan-video on Colab

Text-to-video with Wan 2.1 / 2.2 on ComfyUI.

**Runtime → Change runtime type → T4 GPU** (or better) before running.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Setup (ComfyUI + GGUF nodes)

In [ ]:
%cd /content
!git clone -q https://github.com/Nimra26/wan-video.git
%cd /content/wan-video
!git clone -q https://github.com/comfyanonymous/ComfyUI.git && git -C ComfyUI checkout -q 7a28332
!git clone -q https://github.com/city96/ComfyUI-GGUF.git ComfyUI/custom_nodes/ComfyUI-GGUF && git -C ComfyUI/custom_nodes/ComfyUI-GGUF checkout -q 6ea2651
!pip install -q -r ComfyUI/requirements.txt -r ComfyUI/custom_nodes/ComfyUI-GGUF/requirements.txt

## 2. Download models

Wan 2.1 files are ~3.5 GB. Set `WAN22 = True` to also fetch the Wan 2.2 5B model (~4.5 GB more).

In [ ]:
WAN22 = False  #@param {type:"boolean"}

HF = "https://huggingface.co"
files = {
    "diffusion_models/wan2.1_t2v_1.3B_fp16.safetensors": f"{HF}/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/diffusion_models/wan2.1_t2v_1.3B_fp16.safetensors",
    "vae/wan_2.1_vae.safetensors": f"{HF}/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors",
    "text_encoders/umt5-xxl-encoder-Q4_K_M.gguf": f"{HF}/city96/umt5-xxl-encoder-gguf/resolve/main/umt5-xxl-encoder-Q4_K_M.gguf",
    "upscale_models/RealESRGAN_x2plus.pth": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth",
}
if WAN22:
    files.update({
        "diffusion_models/Wan2.2-TI2V-5B-Q6_K.gguf": f"{HF}/QuantStack/Wan2.2-TI2V-5B-GGUF/resolve/main/Wan2.2-TI2V-5B-Q6_K.gguf",
        "vae/wan2.2_vae.safetensors": f"{HF}/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/vae/wan2.2_vae.safetensors",
    })

import os, subprocess
for rel, url in files.items():
    dst = f"/content/wan-video/ComfyUI/models/{rel}"
    if os.path.exists(dst):
        continue
    print("Downloading", rel)
    subprocess.run(["wget", "-q", "-c", "-O", dst, url], check=True)
print("Models ready.")

## 3. Generate

In [ ]:
PROMPT = "A red fox running through snow at sunrise"  #@param {type:"string"}
MODEL = "wan21"  #@param ["wan21", "wan22"]
SECONDS = 2  #@param {type:"number"}
STEPS = 30  #@param {type:"integer"}

%cd /content/wan-video
!python generate.py "{PROMPT}" --model {MODEL} --seconds {SECONDS} --steps {STEPS}

## 4. Watch / download the latest video

In [ ]:
import glob, os
from base64 import b64encode
from IPython.display import HTML, display
from google.colab import files as colab_files

latest = max(glob.glob("/content/wan-video/outputs/*.mp4"), key=os.path.getmtime)
print(latest)
data = b64encode(open(latest, "rb").read()).decode()
display(HTML(f'<video width="720" controls loop autoplay><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'))
colab_files.download(latest)

If a run fails, check the server log: `!tail -50 /content/wan-video/comfyui.log`